# 00 — Setup and File Discovery

**Purpose:** Before running any analysis, verify that all the input files you need actually exist for this subject.

This notebook does no analysis. It just builds the paths to every file the beta-series pipeline will use, checks that each file is present on disk, and prints a clear summary. If anything is missing, you want to know now rather than halfway through a long model fit.

**Files you need per subject:**
- Preprocessed BOLD images (one per task run, from fMRIPrep)
- Brain masks (one per task run, from fMRIPrep)
- Confounds timeseries (one per task run, from fMRIPrep)
- Events TSV files (one per task run, from the raw BIDS directory)

**Output of this notebook:** Nothing is saved. It is a sanity check only.

---
## Section 1 — Configuration

This is the **only cell you need to change** when switching subjects. Set the subject ID here, and all paths below will update automatically.

**TODO:**
- [ ] Set `subject` to the subject ID string (e.g., `"sub-097"`)
- [ ] Set `tasks` to the list of task run names (e.g., `["modulate1", "modulate2"]`)

In [ ]:
subject = "sub-097"
tasks = ["modulate1", "modulate2"]

---
## Section 2 — Build Directory Paths

Build everything from `project_dir` so changing `subject` in Section 1 flows through automatically. The `/` operator on `Path` objects joins directories — it's equivalent to `os.path.join`.

```python
project_dir     = Path(r"C:\ManzaRotation")
raw_dir         = project_dir / "Raw"
derivatives_dir = project_dir / "Derivatives"
analysis_dir    = project_dir / "Analysis"

# Subject-specific subdirectories
subject_deriv_dir   = derivatives_dir / subject          # .../Derivatives/sub-097
subject_raw_dir     = raw_dir / subject                  # .../Raw/sub-097
func_deriv_dir      = subject_deriv_dir / "func"         # .../Derivatives/sub-097/func
anat_deriv_dir      = subject_deriv_dir / "anat"         # .../Derivatives/sub-097/anat

beta_series_out_dir = analysis_dir / "outputs" / subject / "beta_series"
```

**TODO:**
- [ ] Define all of the above as `Path` objects
- [ ] Print each one so you can visually confirm the strings look right

In [ ]:
from pathlib import Path

# TODO: define project_dir and derived directories
# project_dir = Path(r"C:\ManzaRotation")
# ...


---
## Section 3 — Find and Check BOLD Files

The preprocessed BOLD images live in the fMRIPrep derivatives folder under `func/`. For `sub-097`, the two full paths are:

```
C:\ManzaRotation\Derivatives\sub-097\func\sub-097_task-modulate1_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz
C:\ManzaRotation\Derivatives\sub-097\func\sub-097_task-modulate2_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz
```

Loop over `tasks`, build the path with an f-string, and check `.exists()`:

```python
for task in tasks:
    bold_path = func_deriv_dir / f"{subject}_task-{task}_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz"
    status = "OK" if bold_path.exists() else "MISSING"
    print(f"  [{status}] {bold_path}")
```

**TODO:**
- [ ] Write this loop (you can copy the pattern above directly)
- [ ] Confirm both files print as `OK` before moving on

In [ ]:
# TODO: loop over tasks, build BOLD path, check .exists(), print result


---
## Section 4 — Find and Check Brain Mask Files

fMRIPrep outputs a brain mask for each run alongside the BOLD. For `sub-097`, the two full paths are:

```
C:\ManzaRotation\Derivatives\sub-097\func\sub-097_task-modulate1_space-MNI152NLin2009cAsym_res-2_desc-brain_mask.nii.gz
C:\ManzaRotation\Derivatives\sub-097\func\sub-097_task-modulate2_space-MNI152NLin2009cAsym_res-2_desc-brain_mask.nii.gz
```

Same loop pattern as Section 3, just swap in the mask filename suffix:

```python
for task in tasks:
    mask_path = func_deriv_dir / f"{subject}_task-{task}_space-MNI152NLin2009cAsym_res-2_desc-brain_mask.nii.gz"
    status = "OK" if mask_path.exists() else "MISSING"
    print(f"  [{status}] {mask_path}")
```

**TODO:**
- [ ] Write this loop
- [ ] Confirm both files print as `OK`

In [ ]:
# TODO: loop over tasks, build mask path, check .exists(), print result


---
## Section 5 — Find and Check Confounds Files

fMRIPrep outputs a confounds timeseries TSV for each run. It has hundreds of columns — motion parameters, CompCor components, cosine drift terms, etc. You'll select a small subset when fitting the GLM. For `sub-097`, the two full paths are:

```
C:\ManzaRotation\Derivatives\sub-097\func\sub-097_task-modulate1_desc-confounds_timeseries.tsv
C:\ManzaRotation\Derivatives\sub-097\func\sub-097_task-modulate2_desc-confounds_timeseries.tsv
```

Same loop pattern again — note the shorter filename (no space/res suffix):

```python
for task in tasks:
    confounds_path = func_deriv_dir / f"{subject}_task-{task}_desc-confounds_timeseries.tsv"
    status = "OK" if confounds_path.exists() else "MISSING"
    print(f"  [{status}] {confounds_path}")
```

**TODO:**
- [ ] Write this loop
- [ ] Confirm both files print as `OK`

In [ ]:
# TODO: loop over tasks, build confounds path, check .exists(), print result


---
## Section 6 — Find and Check Events Files

The events TSV files are in the **raw** BIDS directory (not derivatives). For `sub-097`, the two full paths are:

```
C:\ManzaRotation\Raw\sub-097\func\sub-097_task-modulate1_events.tsv
C:\ManzaRotation\Raw\sub-097\func\sub-097_task-modulate2_events.tsv
```

Same loop pattern — but the base directory is `subject_raw_dir / "func"`, not `func_deriv_dir`:

```python
raw_func_dir = subject_raw_dir / "func"

for task in tasks:
    events_path = raw_func_dir / f"{subject}_task-{task}_events.tsv"
    status = "OK" if events_path.exists() else "MISSING"
    print(f"  [{status}] {events_path}")
```

**TODO:**
- [ ] Write this loop
- [ ] Confirm both files print as `OK`

In [ ]:
# TODO: loop over tasks, build events path, check .exists(), print result


---
## Section 7 — Full Summary

Collect all paths into one DataFrame so you can see the full picture at a glance. The pattern is to build a list of dicts — one dict per file — and pass it to `pd.DataFrame`:

```python
rows = []

for task in tasks:
    rows.append({"task": task, "kind": "bold",      "path": str(bold_path),      "exists": bold_path.exists()})
    rows.append({"task": task, "kind": "mask",      "path": str(mask_path),      "exists": mask_path.exists()})
    rows.append({"task": task, "kind": "confounds", "path": str(confounds_path), "exists": confounds_path.exists()})
    rows.append({"task": task, "kind": "events",    "path": str(events_path),    "exists": events_path.exists()})

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
```

You'll need to make sure the path variables are accessible here — either rebuild them inside this loop or store them in a dict as you go through Sections 3–6.

**TODO:**
- [ ] Build `rows` by looping over `tasks` and appending one dict per file type
- [ ] Convert to a DataFrame and print
- [ ] If any row has `exists=False`, stop and investigate before running notebook 01

In [ ]:
import pandas as pd

# TODO: build a list of dicts, one per file, with keys: task, kind, path, exists
# summary = pd.DataFrame([...])
# print(summary.to_string(index=False))
